# AIDA 2156 Final Project – 03 Relationships, Within-Group Comparisons and Uncertainty

**Data version 2** (`shared/synthetic-data/`), cleaning version 1.1, using notebook 02's processing times for the 4,319 requests the team had finished by the extract date.

**Question:** How does processing time relate to task type, missing or conflicting information, team workload, submission timing and intake channel? Do those differences change when groups are compared within a common mix of recorded characteristics, and how uncertain are they?

This is steps 4 and 5 of the final project. Notebook 02 described how long requests took; this notebook compares groups. Initial exploration identified possible differences in group composition (for example, information issues are more common for some intake channels, and team workload differs by task type). The notebook checks whether descriptive comparisons change when the groups are compared within a common mix of recorded characteristics. Some comparisons may simply confirm that a difference is small.

The 13 comparisons that receive confidence intervals were specified before the bootstrap analysis, after the initial exploration. Intervals resample whole employees, because several requests can belong to the same employee. They are descriptive estimates with uncertainty: no p-values are calculated and no difference is called significant.

All records are synthetic. Differences occur in generated data shaped by task-type, workload, timing and delay assumptions; this analysis describes associations, not causes, and says nothing about real employees or real HR performance.

In [1]:
import hashlib
import json
from datetime import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
OUTPUT_DIR = Path("../outputs")
FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(exist_ok=True)

read = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False)
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()

requests = read(SD / "cleaning/output/requests_cleaned.csv")
cleaning_info = json.loads((SD / "cleaning/output/requests_cleaned_info.json").read_text())
assert sha(SD / "cleaning/output/requests_cleaned.csv") == cleaning_info["output_sha256"], \
    "The cleaned file doesn't match its cleaning record"

# Working calendar saved by notebook 02; the generator must be unchanged since then
calendar = json.loads((OUTPUT_DIR / "working_calendar.json").read_text())
assert sha(SD / "requests/generate_requests.py") == calendar["generator_sha256"], \
    "The generator has changed since notebook 02 saved the working calendar"
CLOSURES = {pd.Timestamp(d).date() for d in calendar["closure_dates"]}
DAY_START, DAY_END = time.fromisoformat(calendar["day_start"]), time.fromisoformat(calendar["day_end"])
EXTRACT_AT = pd.Timestamp(calendar["extract_at"])

# Notebook 02's completed requests (processing time in working-day equivalents)
completed = pd.read_csv(OUTPUT_DIR / "processing_times_completed.csv")
groups_02 = pd.read_csv(OUTPUT_DIR / "request_groups.csv", index_col=0)["requests"]
assert completed["request_id"].is_unique and len(completed) == groups_02["completed"], \
    "notebook 02's completed table doesn't match its request groups"
assert completed["info_issue_at_intake"].isin([True, False]).all(), "Unexpected information-issue values"
assert completed["processing_days"].notna().all() and np.isfinite(completed["processing_days"]).all()

# Rebuild notebook 02's eligibility from the current cleaned data: all four stage timestamps present,
# team completion no later than the extract. The saved table must hold exactly those requests,
# with the same employee, task type and information-issue values.
STAGE_TIMES = ["submitted_at", "info_complete_at", "assigned_at", "team_completed_at"]
stage = requests[STAGE_TIMES].apply(lambda c: pd.to_datetime(c.replace("", None), format="%Y-%m-%dT%H:%M"))
eligible = requests.loc[stage.notna().all(axis=1) & (stage["team_completed_at"] <= EXTRACT_AT), "request_id"]
assert set(completed["request_id"]) == set(eligible), "notebook 02's completed requests differ from the rebuilt eligibility"
source = requests.set_index("request_id").loc[completed["request_id"]]
assert (completed["employee_id"].to_numpy() == source["employee_id"].to_numpy()).all(), "Employee IDs differ"
assert (completed["task_type"].to_numpy() == source["task_type"].to_numpy()).all(), "Task types differ"
assert (completed["info_issue_at_intake"].astype(str).to_numpy() == source["info_issue_at_intake"].to_numpy()).all(), \
    "Information-issue values differ"

# Record which inputs this analysis used
inputs_record = {"data_version": DATA_VERSION, "cleaning_version": cleaning_info["cleaning_version"],
                 "inputs_sha256": {"requests_cleaned.csv": cleaning_info["output_sha256"],
                                   "processing_times_completed.csv (notebook 02)": sha(OUTPUT_DIR / "processing_times_completed.csv"),
                                   "working_calendar.json (notebook 02)": sha(OUTPUT_DIR / "working_calendar.json")}}
(OUTPUT_DIR / "analysis_inputs_03.json").write_text(json.dumps(inputs_record, indent=2))

print(f"Data version {DATA_VERSION}; cleaning version {cleaning_info['cleaning_version']}; extract {EXTRACT_AT:%Y-%m-%d %H:%M}")
print(f"{len(completed):,} completed requests from {completed['employee_id'].nunique():,} employees "
      f"({len(requests):,} cleaned requests in total)")

Data version 2; cleaning version 1.1; extract 2026-09-29 17:00
4,319 completed requests from 1,065 employees (4,364 cleaned requests in total)


## A. Definitions

**Processing time** is notebook 02's measure: working time from submission to team completion, in working-day equivalents (working hours ÷ 8.5).

**Variables compared**

- **Task type, intake channel, hiring department, employee type, information issue at intake:** recorded on the cleaned request. A missing intake channel is kept as its own group, `unknown`, and shown in every table; it is left out only of the phone and email comparisons with web forms.
- **After-hours submission:** submitted on a weekend, on a simulated closure date, or before 08:00 or after 16:30 on a working day. Exactly 08:00 and 16:30 count as working time, matching the generator's rule for working moments. The calendar is the one notebook 02 saved.
- **Workload at assignment:** the number of other tasks for the same team that were assigned strictly before this task was assigned and finished strictly after it (or were still unfinished at the extract). This follows the data card's definition and uses all cleaned requests. It is not a characteristic of the request at submission: it reflects timing, task mix and workflow conditions.
- **Workload groups:** the cutoffs are the lower and upper quartiles of workload across all 4,319 completed requests. **Low** is at or below the lower quartile and **high** is at or above the upper quartile, so tied values join the outer group and each group can hold more than a quarter of requests; the sizes are reported. The same overall cutoffs are used inside each task type.
- **Spring:** March to May, by submission date.

**The 13 contrasts with intervals** (always *comparison group minus reference group*, in working-day equivalents of median processing time):

| # | Comparison − reference | Summary |
|---|---|---|
| 1–4 | Equipment setup, system access, workplace access, orientation scheduling − payroll setup | Raw median difference |
| 5 | Information issue − no information issue | Raw median difference |
| 6 | High − low workload | Raw median difference |
| 7 | High − low workload, within task type | Weighted within-stratum median difference |
| 8 | After-hours − working-hours submission | Raw median difference |
| 9 | After-hours − working-hours, within information issue × task type | Weighted within-stratum median difference |
| 10 | Phone − web form | Raw median difference |
| 11 | Phone − web form, within information issue | Weighted within-stratum median difference |
| 12 | Email − web form | Raw median difference |
| 13 | Email − web form, within information issue | Weighted within-stratum median difference |

Payroll setup is the reference task type because it is operationally meaningful: every new employee needs pay set up before their first pay date, so it is a natural baseline for the other setup tasks. It is also the largest task group.

**Weighted within-stratum median difference.** Within each stratum (for example, each task type), the median of the comparison group minus the median of the reference group; those differences are then averaged using fixed weights. The common target is **all 4,319 completed requests**: each stratum's weight is its share of all completed requests. A stratum is included in the weighted estimate when each comparison group has at least 5 requests in it. This is a calculation rule, not a guarantee of precision; request and employee counts are reported for both sides. Strata that don't meet it are listed and left out, the weights are rescaled once over the included strata, and the share of all completed requests retained (**coverage**) is reported; the estimate then applies to the included strata only. This is a descriptive summary, not the difference between two standardised population medians and not a causal effect. The raw median difference is always shown beside it, because the two answer different questions.

**Interpretation limits.** Information issues are directly connected to the information-check stage, so their total-duration difference largely reflects that stage; it does not on its own show an effect on later team processing. Workload at assignment is an association measure, and comparing within task type does not make it causal.

In [2]:
r = requests.copy()
for c in ["submitted_at", "assigned_at", "team_completed_at"]:
    r[c] = pd.to_datetime(r[c].replace("", None), format="%Y-%m-%dT%H:%M")

# Workload at assignment (data card definition), using all cleaned requests with an assignment time
assigned = r[r["assigned_at"].notna()]
workload = {}
for team, g in assigned.groupby("responsible_team"):
    a = g["assigned_at"].to_numpy()
    done = g["team_completed_at"].fillna(pd.Timestamp.max).to_numpy()       # unfinished: still open after any time
    for rid, t in zip(g["request_id"], a):
        workload[rid] = int(((a < t) & (done > t)).sum())

d = completed.merge(r[["request_id", "employee_id", "responsible_team", "intake_channel", "hiring_department",
                       "employee_type", "submitted_at", "assigned_at"]],
                    on="request_id", how="left", validate="one_to_one", suffixes=("", "_cleaned"))
assert (d["employee_id"] == d["employee_id_cleaned"]).all(), "Employee IDs differ from the cleaned requests"
d = d.drop(columns="employee_id_cleaned")
d["intake_channel"] = d["intake_channel"].replace("", "unknown")
d["workload_at_assignment"] = d["request_id"].map(workload)
assert d["workload_at_assignment"].notna().all(), "Every completed request should have a workload at assignment"

s = d["submitted_at"]
assert s.notna().all(), "Every included request needs a submission timestamp"
assert s.le(EXTRACT_AT).all(), "A submission occurs after extraction"
in_hours = s.dt.time.between(DAY_START, DAY_END, inclusive="both")    # 08:00 and 16:30 count as working time
d["after_hours"] = (s.dt.weekday >= 5) | s.dt.date.isin(CLOSURES) | ~in_hours
d["spring"] = s.dt.month.isin([3, 4, 5])

Q1, Q3 = d["workload_at_assignment"].quantile([0.25, 0.75])
assert Q1 < Q3, "Workload quartiles coincide; low and high groups need a different definition"
d["workload_group"] = np.select([d["workload_at_assignment"] <= Q1, d["workload_at_assignment"] >= Q3],
                                ["low", "high"], default="middle")
print(f"Workload at assignment: quartile cutoffs {Q1:g} and {Q3:g} open tasks; "
      f"groups: {d['workload_group'].value_counts().reindex(['low', 'middle', 'high']).to_dict()}")
print(f"After-hours submissions: {int(d['after_hours'].sum()):,} of {len(d):,} completed requests "
      f"({100 * d['after_hours'].mean():.1f}%)")
print(f"Unknown intake channel: {int((d['intake_channel'] == 'unknown').sum())} completed requests "
      "(kept as a group; excluded only from the phone/email comparisons)")

Workload at assignment: quartile cutoffs 9 and 22 open tasks; groups: {'low': 1160, 'middle': 2027, 'high': 1132}
After-hours submissions: 650 of 4,319 completed requests (15.0%)
Unknown intake channel: 44 completed requests (kept as a group; excluded only from the phone/email comparisons)
